In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit1025.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit1047.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit486.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit728.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit1013.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit282.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit687.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit889.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit554.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit772.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit305.txt
/kaggle/input/yol

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 49.8 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall

In [3]:
from ultralytics import YOLO
import time, pandas as pd, os, json
from tabulate import tabulate

# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# baseline params (fixed across ablations, Case 1)
base_params = {
    "batch": 32,
    "lr0": 0.01,
    "optimizer": "SGD",
    "imgsz": 640,
    "weight_decay": 0.0005,
    "dropout": 0.0,        # no dropout
    "freeze": 0,           # no frozen layers
    "activation": "SiLU"   # default activation
}

# results log file (specific to Case 1)
results_file = "case1_results.csv"

# initialize CSV if not exists
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", 
        "Epochs", 
        "Training Time (s)", 
        "Precision", 
        "Recall", 
        "mAP@0.5", 
        "mAP@0.5:0.95", 
        "Parameters"
    ]).to_csv(results_file, index=False)

def log_result(experiment, epochs, results, train_time):
    """Append YOLO training results to Case 1 CSV"""
    df = pd.read_csv(results_file)

    new_row = {
        "Experiment": experiment,
        "Epochs": epochs,
        "Training Time (s)": round(train_time, 2),
        "Precision": results.results_dict["metrics/precision(B)"],
        "Recall": results.results_dict["metrics/recall(B)"],
        "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
        "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        "Parameters": json.dumps(base_params)
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# =============================
# Case 1: Epoch variations
# =============================

epoch_variations = [16, 32, 64]

all_results = []

for ep in epoch_variations:
    print(f"\n🚀 Case 1: Training YOLOv8s for {ep} epochs...")
    model = YOLO("yolov8s.pt")

    start_time = time.time()
    results = model.train(
        data=data_yaml,
        epochs=ep,
        batch=base_params["batch"],
        lr0=base_params["lr0"],
        optimizer=base_params["optimizer"],
        imgsz=base_params["imgsz"],
        weight_decay=base_params["weight_decay"],
        dropout=base_params["dropout"],
        freeze=base_params["freeze"]
    )
    end_time = time.time()

    log_result(f"case1_epochs_{ep}", ep, results, end_time - start_time)

    all_results.append([
        f"case1_epochs_{ep}",
        ep,
        round(end_time - start_time, 2),
        results.results_dict["metrics/precision(B)"],
        results.results_dict["metrics/recall(B)"],
        results.results_dict["metrics/mAP50(B)"],
        results.results_dict["metrics/mAP50-95(B)"],
    ])

# =============================
# Print pretty results table
# =============================
headers = ["Experiment", "Epochs", "Train Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"]
print("\n📊 Case 1 Results Table:\n")
print(tabulate(all_results, headers=headers, tablefmt="grid"))

print(f"\n✅ Case 1 completed. Results saved to {results_file}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🚀 Case 1: Training YOLOv8s for 16 epochs...


Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=16, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100, pers

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5.0±1.9 MB/s, size: 51.9 KB)


train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:04<00:00, 184.04it/s]

...

      64/64      7.87G     0.8053     0.3192       1.06        125        640: 100%|██████████| 29/29 [00:16<00:00,  1.75it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.30it/s]

                   all        230       1440      0.988      0.987      0.986      0.682



64 epochs completed in 0.334 hours.
Optimizer stripped from runs/detect/train3/weights/last.pt, 22.5MB
Optimizer stripped from runs/detect/train3/weights/best.pt, 22.5MB

Validating runs/detect/train3/weights/best.pt...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:03<00:00,  1.06it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.987      0.987      0.988      0.686
     DC Voltage Source        103        144      0.992      0.972      0.984      0.768
              Resistor        183        605      0.983      0.984      0.988      0.656
        Current Source        113        150       0.98          1      0.981      0.714
              Inductor        117        177      0.992      0.989      0.993      0.637
             Capacitor        115        184      0.983      0.989      0.989      0.605
     AC Voltage Source         65        180      0.994       0.99      0.993      0.736
Speed: 0.5ms preprocess, 4.1ms inference, 0.0ms loss, 5.7ms postprocess per image
Results saved to runs/detect/train3

📊 Case 1 Results Table:

+-----------------+----------+------------------+-------------+----------+-----------+----------------+
| Experiment      |   Epochs |   Train Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+=================+======

In [4]:
from ultralytics import YOLO
import time, pandas as pd, os, json
from tabulate import tabulate

# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# baseline params (fixed across cases)
base_params = {
    "batch": 32,
    "lr0": 0.01,
    "optimizer": "SGD",
    "imgsz": 640,
    "epochs": 32,          # fixed from Case 1 best
    "dropout": 0.0,
    "freeze": 0,
    "activation": "SiLU"
}

# results log file (continuing from Case 1)
results_file = "case1_results.csv"

# make sure CSV exists (should already from Case 1)
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", 
        "Epochs", 
        "Training Time (s)", 
        "Precision", 
        "Recall", 
        "mAP@0.5", 
        "mAP@0.5:0.95", 
        "Parameters"
    ]).to_csv(results_file, index=False)

def log_result(experiment, weight_decay, results, train_time):
    """Append YOLO training results to CSV"""
    df = pd.read_csv(results_file)

    params = base_params.copy()
    params["weight_decay"] = weight_decay

    new_row = {
        "Experiment": experiment,
        "Epochs": params["epochs"],
        "Training Time (s)": round(train_time, 2),
        "Precision": results.results_dict["metrics/precision(B)"],
        "Recall": results.results_dict["metrics/recall(B)"],
        "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
        "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        "Parameters": json.dumps(params)
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# =============================
# Case 2: Weight Decay variations
# =============================

weight_decay_variations = [0, 0.0001, 0.0005, 0.001, 0.005]
all_results = []

for wd in weight_decay_variations:
    print(f"\n🚀 Case 2: Training YOLOv8s with weight_decay={wd}, epochs={base_params['epochs']}...")
    model = YOLO("yolov8s.pt")

    start_time = time.time()
    results = model.train(
        data=data_yaml,
        epochs=base_params["epochs"],
        batch=base_params["batch"],
        lr0=base_params["lr0"],
        optimizer=base_params["optimizer"],
        imgsz=base_params["imgsz"],
        weight_decay=wd,
        dropout=base_params["dropout"],
        freeze=base_params["freeze"]
    )
    end_time = time.time()

    log_result(f"case2_wd_{wd}", wd, results, end_time - start_time)

    all_results.append([
        f"case2_wd_{wd}",
        base_params["epochs"],
        round(end_time - start_time, 2),
        results.results_dict["metrics/precision(B)"],
        results.results_dict["metrics/recall(B)"],
        results.results_dict["metrics/mAP50(B)"],
        results.results_dict["metrics/mAP50-95(B)"],
    ])

# =============================
# Print pretty results table
# =============================
headers = ["Experiment", "Epochs", "Train Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"]
print("\n📊 Case 2 Results Table:\n")
print(tabulate(all_results, headers=headers, tablefmt="grid"))

print(f"\n✅ Case 2 completed. Results appended to {results_file}")



🚀 Case 2: Training YOLOv8s with weight_decay=0, epochs=32...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optim

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 769.89it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 42.3±26.6 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 468.51it/s]

...

      32/32      7.85G      1.006     0.3737      1.178        142        640: 100%|██████████| 29/29 [00:16<00:00,  1.79it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.44it/s]

                   all        230       1440      0.986      0.985      0.988      0.682



32 epochs completed in 0.167 hours.
Optimizer stripped from runs/detect/train8/weights/last.pt, 22.5MB
Optimizer stripped from runs/detect/train8/weights/best.pt, 22.5MB

Validating runs/detect/train8/weights/best.pt...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:03<00:00,  1.06it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.987      0.986      0.988      0.686
     DC Voltage Source        103        144          1      0.967      0.983      0.767
              Resistor        183        605      0.984      0.992      0.985      0.641
        Current Source        113        150      0.963      0.993      0.986      0.735
              Inductor        117        177      0.994      0.989      0.989      0.644
             Capacitor        115        184      0.994      0.989       0.99      0.609
     AC Voltage Source         65        180      0.988      0.989      0.991      0.718
Speed: 0.4ms preprocess, 3.7ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to runs/detect/train8

📊 Case 2 Results Table:

+-----------------+----------+------------------+-------------+----------+-----------+----------------+
| Experiment      |   Epochs |   Train Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+=================+======

In [5]:
from ultralytics import YOLO
import time, pandas as pd, os, json
from tabulate import tabulate

# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# baseline params (fixed across cases)
base_params = {
    "lr0": 0.01,
    "optimizer": "SGD",
    "imgsz": 640,
    "epochs": 32,           # fixed from Case 1
    "weight_decay": 0.001,  # fixed from Case 2
    "dropout": 0.0,
    "freeze": 0,
    "activation": "SiLU"
}

# results log file (append cases together)
results_file = "case1_results.csv"

# ensure CSV exists
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", 
        "Epochs", 
        "Batch Size",
        "Training Time (s)", 
        "Precision", 
        "Recall", 
        "mAP@0.5", 
        "mAP@0.5:0.95", 
        "Parameters"
    ]).to_csv(results_file, index=False)

def log_result(experiment, batch_size, results, train_time):
    """Append YOLO training results to CSV"""
    df = pd.read_csv(results_file)

    params = base_params.copy()
    params["batch"] = batch_size

    new_row = {
        "Experiment": experiment,
        "Epochs": params["epochs"],
        "Batch Size": batch_size,
        "Training Time (s)": round(train_time, 2),
        "Precision": results.results_dict["metrics/precision(B)"],
        "Recall": results.results_dict["metrics/recall(B)"],
        "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
        "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        "Parameters": json.dumps(params)
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# =============================
# Case 3: Batch Size variations
# =============================

batch_variations = [16, 32, 64]
all_results = []

for bs in batch_variations:
    print(f"\n🚀 Case 3: Training YOLOv8s with batch_size={bs}, epochs={base_params['epochs']}, weight_decay={base_params['weight_decay']}...")
    model = YOLO("yolov8s.pt")

    start_time = time.time()
    results = model.train(
        data=data_yaml,
        epochs=base_params["epochs"],
        batch=bs,
        lr0=base_params["lr0"],
        optimizer=base_params["optimizer"],
        imgsz=base_params["imgsz"],
        weight_decay=base_params["weight_decay"],
        dropout=base_params["dropout"],
        freeze=base_params["freeze"]
    )
    end_time = time.time()

    log_result(f"case3_bs_{bs}", bs, results, end_time - start_time)

    all_results.append([
        f"case3_bs_{bs}",
        base_params["epochs"],
        bs,
        round(end_time - start_time, 2),
        results.results_dict["metrics/precision(B)"],
        results.results_dict["metrics/recall(B)"],
        results.results_dict["metrics/mAP50(B)"],
        results.results_dict["metrics/mAP50-95(B)"],
    ])

# =============================
# Print pretty results table
# =============================
headers = ["Experiment", "Epochs", "Batch Size", "Train Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"]
print("\n📊 Case 3 Results Table:\n")
print(tabulate(all_results, headers=headers, tablefmt="grid"))

print(f"\n✅ Case 3 completed. Results appended to {results_file}")



🚀 Case 3: Training YOLOv8s with batch_size=16, epochs=32, weight_decay=0.001...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train9, nbs=64, nms=False

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 812.62it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 68.2±25.0 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 484.26it/s]

...

      32/32      15.2G      1.009     0.3792      1.183        142        640: 100%|██████████| 15/15 [00:15<00:00,  1.03s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:01<00:00,  1.12it/s]

                   all        230       1440      0.986      0.987      0.986      0.693



32 epochs completed in 0.165 hours.
Optimizer stripped from runs/detect/train11/weights/last.pt, 22.5MB
Optimizer stripped from runs/detect/train11/weights/best.pt, 22.5MB

Validating runs/detect/train11/weights/best.pt...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:02<00:00,  1.19s/it]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.985      0.987      0.986      0.693
     DC Voltage Source        103        144      0.993      0.972      0.982      0.775
              Resistor        183        605      0.984       0.99      0.987      0.655
        Current Source        113        150      0.968      0.999      0.976      0.726
              Inductor        117        177      0.992      0.989       0.99      0.651
             Capacitor        115        184      0.982      0.989      0.988      0.621
     AC Voltage Source         65        180      0.994      0.984      0.992       0.73
Speed: 0.4ms preprocess, 4.2ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs/detect/train11

📊 Case 3 Results Table:

+--------------+----------+--------------+------------------+-------------+----------+-----------+----------------+
| Experiment   |   Epochs |   Batch Size |   Train Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |


In [6]:
from ultralytics import YOLO
import time, pandas as pd, os, json
from tabulate import tabulate

# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# baseline params (fixed across cases)
base_params = {
    "batch": 32,           # fixed from Case 3
    "lr0": 0.01,
    "optimizer": "SGD",
    "imgsz": 640,
    "epochs": 32,          # fixed from Case 1
    "weight_decay": 0.001, # fixed from Case 2
    "freeze": 0,
    "activation": "SiLU"
}

# results log file (continue appending)
results_file = "case1_results.csv"

# ensure CSV exists
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", 
        "Epochs", 
        "Batch Size",
        "Dropout",
        "Training Time (s)", 
        "Precision", 
        "Recall", 
        "mAP@0.5", 
        "mAP@0.5:0.95", 
        "Parameters"
    ]).to_csv(results_file, index=False)

def log_result(experiment, dropout, results, train_time):
    """Append YOLO training results to CSV"""
    df = pd.read_csv(results_file)

    params = base_params.copy()
    params["dropout"] = dropout

    new_row = {
        "Experiment": experiment,
        "Epochs": params["epochs"],
        "Batch Size": params["batch"],
        "Dropout": dropout,
        "Training Time (s)": round(train_time, 2),
        "Precision": results.results_dict["metrics/precision(B)"],
        "Recall": results.results_dict["metrics/recall(B)"],
        "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
        "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        "Parameters": json.dumps(params)
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# =============================
# Case 4: Dropout variations
# =============================

dropout_variations = [0, 0.1, 0.2, 0.3, 0.5]
all_results = []

for dr in dropout_variations:
    print(f"\n🚀 Case 4: Training YOLOv8s with dropout={dr}, batch_size={base_params['batch']}, epochs={base_params['epochs']}, weight_decay={base_params['weight_decay']}...")
    model = YOLO("yolov8s.pt")

    start_time = time.time()
    results = model.train(
        data=data_yaml,
        epochs=base_params["epochs"],
        batch=base_params["batch"],
        lr0=base_params["lr0"],
        optimizer=base_params["optimizer"],
        imgsz=base_params["imgsz"],
        weight_decay=base_params["weight_decay"],
        dropout=dr,
        freeze=base_params["freeze"]
    )
    end_time = time.time()

    log_result(f"case4_dropout_{dr}", dr, results, end_time - start_time)

    all_results.append([
        f"case4_dropout_{dr}",
        base_params["epochs"],
        base_params["batch"],
        dr,
        round(end_time - start_time, 2),
        results.results_dict["metrics/precision(B)"],
        results.results_dict["metrics/recall(B)"],
        results.results_dict["metrics/mAP50(B)"],
        results.results_dict["metrics/mAP50-95(B)"],
    ])

# =============================
# Print pretty results table
# =============================
headers = ["Experiment", "Epochs", "Batch Size", "Dropout", "Train Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"]
print("\n📊 Case 4 Results Table:\n")
print(tabulate(all_results, headers=headers, tablefmt="grid"))

print(f"\n✅ Case 4 completed. Results appended to {results_file}")



🚀 Case 4: Training YOLOv8s with dropout=0, batch_size=32, epochs=32, weight_decay=0.001...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train12, nbs=64,

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 745.62it/s]


WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 66.4±25.0 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 454.38it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.


...

      32/32      7.85G     0.9993     0.3736      1.175        142        640: 100%|██████████| 29/29 [00:16<00:00,  1.73it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.27it/s]

                   all        230       1440      0.985      0.988      0.988      0.691



32 epochs completed in 0.169 hours.
Optimizer stripped from runs/detect/train16/weights/last.pt, 22.5MB
Optimizer stripped from runs/detect/train16/weights/best.pt, 22.5MB

Validating runs/detect/train16/weights/best.pt...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:03<00:00,  1.16it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.985      0.988      0.988      0.691
     DC Voltage Source        103        144      0.987      0.972      0.982      0.769
              Resistor        183        605      0.985      0.988      0.986      0.656
        Current Source        113        150      0.974      0.999      0.988      0.725
              Inductor        117        177      0.987      0.989      0.992      0.655
             Capacitor        115        184      0.982      0.989      0.987      0.617
     AC Voltage Source         65        180      0.994      0.989      0.991      0.724
Speed: 0.5ms preprocess, 3.7ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to runs/detect/train16

📊 Case 4 Results Table:

+-------------------+----------+--------------+-----------+------------------+-------------+----------+-----------+----------------+
| Experiment        |   Epochs |   Batch Size |   Dropout |   Train Time (s) |   Precision |   Reca

In [7]:
from ultralytics import YOLO
import time, pandas as pd, os, json
from tabulate import tabulate

# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# baseline params (fixed from previous best)
base_params = {
    "batch": 32,
    "lr0": 0.01,
    "optimizer": "SGD",
    "epochs": 32,
    "weight_decay": 0.001,
    "dropout": 0.0,
    "freeze": 0,
    "activation": "SiLU"
}

# results log file (one big file for all cases)
results_file = "case1_results.csv"

# ensure CSV exists
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", 
        "Epochs", 
        "Batch Size",
        "Image Size",
        "Training Time (s)", 
        "Precision", 
        "Recall", 
        "mAP@0.5", 
        "mAP@0.5:0.95", 
        "Parameters"
    ]).to_csv(results_file, index=False)

def log_result(experiment, imgsz, results, train_time):
    """Append YOLO training results to CSV"""
    df = pd.read_csv(results_file)

    params = base_params.copy()
    params["imgsz"] = imgsz

    new_row = {
        "Experiment": experiment,
        "Epochs": params["epochs"],
        "Batch Size": params["batch"],
        "Image Size": imgsz,
        "Training Time (s)": round(train_time, 2),
        "Precision": results.results_dict["metrics/precision(B)"],
        "Recall": results.results_dict["metrics/recall(B)"],
        "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
        "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        "Parameters": json.dumps(params)
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# =============================
# Case 5: Image Size variations
# =============================

img_variations = [320, 512, 640]
all_results = []

for imgsz in img_variations:
    print(f"\n🚀 Case 5: Training YOLOv8s with image size={imgsz}, batch={base_params['batch']}, epochs={base_params['epochs']}...")
    model = YOLO("yolov8s.pt")

    start_time = time.time()
    results = model.train(
        data=data_yaml,
        epochs=base_params["epochs"],
        batch=base_params["batch"],
        lr0=base_params["lr0"],
        optimizer=base_params["optimizer"],
        imgsz=imgsz,
        weight_decay=base_params["weight_decay"],
        dropout=base_params["dropout"],
        freeze=base_params["freeze"]
    )
    end_time = time.time()

    log_result(f"case5_imgsz_{imgsz}", imgsz, results, end_time - start_time)

    all_results.append([
        f"case5_imgsz_{imgsz}",
        base_params["epochs"],
        base_params["batch"],
        imgsz,
        round(end_time - start_time, 2),
        results.results_dict["metrics/precision(B)"],
        results.results_dict["metrics/recall(B)"],
        results.results_dict["metrics/mAP50(B)"],
        results.results_dict["metrics/mAP50-95(B)"],
    ])

# =============================
# Print pretty results table
# =============================
headers = ["Experiment", "Epochs", "Batch Size", "Image Size", "Train Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"]
print("\n📊 Case 5 Results Table:\n")
print(tabulate(all_results, headers=headers, tablefmt="grid"))

print(f"\n✅ Case 5 completed. Results appended to {results_file}")



🚀 Case 5: Training YOLOv8s with image size=320, batch=32, epochs=32...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train17, nbs=64, nms=False, opset=

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 704.13it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 47.7±24.2 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 467.29it/s]

...

      32/32      7.95G     0.9993     0.3736      1.175        142        640: 100%|██████████| 29/29 [00:16<00:00,  1.79it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.41it/s]

                   all        230       1440      0.985      0.988      0.988      0.691



32 epochs completed in 0.165 hours.
Optimizer stripped from runs/detect/train19/weights/last.pt, 22.5MB
Optimizer stripped from runs/detect/train19/weights/best.pt, 22.5MB

Validating runs/detect/train19/weights/best.pt...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:03<00:00,  1.25it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.985      0.988      0.988      0.691
     DC Voltage Source        103        144      0.987      0.972      0.982      0.769
              Resistor        183        605      0.985      0.988      0.986      0.656
        Current Source        113        150      0.974      0.999      0.988      0.725
              Inductor        117        177      0.987      0.989      0.992      0.655
             Capacitor        115        184      0.982      0.989      0.987      0.617
     AC Voltage Source         65        180      0.994      0.989      0.991      0.724
Speed: 0.3ms preprocess, 3.7ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to runs/detect/train19

📊 Case 5 Results Table:

+-----------------+----------+--------------+--------------+------------------+-------------+----------+-----------+----------------+
| Experiment      |   Epochs |   Batch Size |   Image Size |   Train Time (s) |   Precision |   Re

In [ ]:
from ultralytics import YOLO
import time, pandas as pd, os, json
from tabulate import tabulate

# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# baseline params (fixed from previous bests)
base_params = {
    "batch": 32,
    "lr0": 0.01,
    "epochs": 32,
    "weight_decay": 0.001,
    "dropout": 0.0,
    "freeze": 0,
    "activation": "SiLU",
    "imgsz": 640   # fixed from Case 5
}

# results log file (big one file for all cases)
results_file = "case1_results.csv"

# ensure CSV exists
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", 
        "Epochs", 
        "Batch Size",
        "Optimizer",
        "Image Size",
        "Training Time (s)", 
        "Precision", 
        "Recall", 
        "mAP@0.5", 
        "mAP@0.5:0.95", 
        "Parameters"
    ]).to_csv(results_file, index=False)

def log_result(experiment, optimizer, results, train_time):
    """Append YOLO training results to CSV"""
    df = pd.read_csv(results_file)

    params = base_params.copy()
    params["optimizer"] = optimizer

    new_row = {
        "Experiment": experiment,
        "Epochs": params["epochs"],
        "Batch Size": params["batch"],
        "Optimizer": optimizer,
        "Image Size": params["imgsz"],
        "Training Time (s)": round(train_time, 2),
        "Precision": results.results_dict["metrics/precision(B)"],
        "Recall": results.results_dict["metrics/recall(B)"],
        "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
        "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        "Parameters": json.dumps(params)
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# =============================
# Case 6: Optimizer variations
# =============================

optim_variations = ["SGD", "Adam", "AdamW", "RMSProp", "Adamax"]
all_results = []

for opt in optim_variations:
    print(f"\n🚀 Case 6: Training YOLOv8s with optimizer={opt}, epochs={base_params['epochs']}, batch={base_params['batch']}, imgsz={base_params['imgsz']}...")
    model = YOLO("yolov8s.pt")

    start_time = time.time()
    results = model.train(
        data=data_yaml,
        epochs=base_params["epochs"],
        batch=base_params["batch"],
        lr0=base_params["lr0"],
        optimizer=opt,
        imgsz=base_params["imgsz"],
        weight_decay=base_params["weight_decay"],
        dropout=base_params["dropout"],
        freeze=base_params["freeze"]
    )
    end_time = time.time()

    log_result(f"case6_opt_{opt}", opt, results, end_time - start_time)

    all_results.append([
        f"case6_opt_{opt}",
        base_params["epochs"],
        base_params["batch"],
        opt,
        base_params["imgsz"],
        round(end_time - start_time, 2),
        results.results_dict["metrics/precision(B)"],
        results.results_dict["metrics/recall(B)"],
        results.results_dict["metrics/mAP50(B)"],
        results.results_dict["metrics/mAP50-95(B)"],
    ])

# =============================
# Print pretty results table
# =============================
headers = ["Experiment", "Epochs", "Batch Size", "Optimizer", "Image Size", "Train Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"]
print("\n📊 Case 6 Results Table:\n")
print(tabulate(all_results, headers=headers, tablefmt="grid"))

print(f"\n✅ Case 6 completed. Results appended to {results_file}")



🚀 Case 6: Training YOLOv8s with optimizer=SGD, epochs=32, batch=32, imgsz=640...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train20, nbs=64, nms=Fal

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 670.30it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 53.7±24.6 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 414.53it/s]

...

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 52.4±32.5 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 526.41it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.


Plotting labels to runs/detect/train24/labels.jpg... 
optimizer: Adamax(lr=0.01, momentum=0.937) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.001), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to runs/detect/train24
Starting training for 32 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/32      7.87G       3.03      6.569      2.407        306        640:  34%|███▍      | 10/29 [00:06<00:11,  1.71it/s]

In [3]:
from ultralytics import YOLO
import time, pandas as pd, os, json
from tabulate import tabulate

# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# baseline params (fixed from previous bests)
base_params = {
    "batch": 32,
    "lr0": 0.01,
    "epochs": 32,
    "weight_decay": 0.001,
    "dropout": 0.0,
    "freeze": 0,
    "activation": "SiLU",
    "imgsz": 640   # fixed from Case 5
}

# results log file (big one file for all cases)
results_file = "case1_results.csv"

# ensure CSV exists
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", 
        "Epochs", 
        "Batch Size",
        "Optimizer",
        "Image Size",
        "Training Time (s)", 
        "Precision", 
        "Recall", 
        "mAP@0.5", 
        "mAP@0.5:0.95", 
        "Parameters"
    ]).to_csv(results_file, index=False)

def log_result(experiment, optimizer, results, train_time):
    """Append YOLO training results to CSV"""
    df = pd.read_csv(results_file)

    params = base_params.copy()
    params["optimizer"] = optimizer

    new_row = {
        "Experiment": experiment,
        "Epochs": params["epochs"],
        "Batch Size": params["batch"],
        "Optimizer": optimizer,
        "Image Size": params["imgsz"],
        "Training Time (s)": round(train_time, 2),
        "Precision": results.results_dict["metrics/precision(B)"],
        "Recall": results.results_dict["metrics/recall(B)"],
        "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
        "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        "Parameters": json.dumps(params)
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# =============================
# Case 6: Optimizer variations
# =============================

optim_variations = ["SGD", "Adam", "AdamW", "RMSProp", "Adamax"]
all_results = []

for opt in optim_variations:
    print(f"\n🚀 Case 6: Training YOLOv8s with optimizer={opt}, epochs={base_params['epochs']}, batch={base_params['batch']}, imgsz={base_params['imgsz']}...")
    model = YOLO("yolov8s.pt")

    start_time = time.time()
    results = model.train(
        data=data_yaml,
        epochs=base_params["epochs"],
        batch=base_params["batch"],
        lr0=base_params["lr0"],
        optimizer=opt,
        imgsz=base_params["imgsz"],
        weight_decay=base_params["weight_decay"],
        dropout=base_params["dropout"],
        freeze=base_params["freeze"]
    )
    end_time = time.time()

    log_result(f"case6_opt_{opt}", opt, results, end_time - start_time)

    all_results.append([
        f"case6_opt_{opt}",
        base_params["epochs"],
        base_params["batch"],
        opt,
        base_params["imgsz"],
        round(end_time - start_time, 2),
        results.results_dict["metrics/precision(B)"],
        results.results_dict["metrics/recall(B)"],
        results.results_dict["metrics/mAP50(B)"],
        results.results_dict["metrics/mAP50-95(B)"],
    ])

# =============================
# Print pretty results table
# =============================
headers = ["Experiment", "Epochs", "Batch Size", "Optimizer", "Image Size", "Train Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"]
print("\n📊 Case 6 Results Table:\n")
print(tabulate(all_results, headers=headers, tablefmt="grid"))

print(f"\n✅ Case 6 completed. Results appended to {results_file}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🚀 Case 6: Training YOLOv8s with optimizer=SGD, epochs=32, batch=32, imgsz=640...


Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100, pers

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6.6±2.1 MB/s, size: 51.9 KB)


train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:04<00:00, 189.92it/s]


...

      32/32      7.83G      1.113     0.4309      1.214        142        640: 100%|██████████| 29/29 [00:15<00:00,  1.89it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.81it/s]

                   all        230       1440      0.989      0.988      0.989      0.689



32 epochs completed in 0.158 hours.
Optimizer stripped from runs/detect/train5/weights/last.pt, 22.5MB
Optimizer stripped from runs/detect/train5/weights/best.pt, 22.5MB

Validating runs/detect/train5/weights/best.pt...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:02<00:00,  1.40it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.987       0.99      0.989      0.692
     DC Voltage Source        103        144          1      0.977      0.984      0.766
              Resistor        183        605      0.982      0.992      0.987      0.655
        Current Source        113        150       0.98          1       0.99      0.743
              Inductor        117        177      0.986      0.989      0.992      0.648
             Capacitor        115        184      0.982      0.989      0.988      0.613
     AC Voltage Source         65        180      0.994      0.992      0.993      0.724
Speed: 0.1ms preprocess, 3.6ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to runs/detect/train5

📊 Case 6 Results Table:

+-------------------+----------+--------------+-------------+--------------+------------------+-------------+----------+-----------+----------------+
| Experiment        |   Epochs |   Batch Size | Optimizer   |   Image Size |   Trai

In [4]:
from ultralytics import YOLO
import time, pandas as pd, os, json

# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# best baseline params (from Cases 1–5)
base_params = {
    "batch": 32,
    "epochs": 32,
    "weight_decay": 0.001,
    "dropout": 0.0,
    "freeze": 0,
    "activation": "SiLU",
    "imgsz": 640
}

# results log file (same master file)
results_file = "case1_results.csv"

def log_result(experiment, optimizer, lr0, results, train_time):
    df = pd.read_csv(results_file)

    params = base_params.copy()
    params["optimizer"] = optimizer
    params["lr0"] = lr0

    new_row = {
        "Experiment": experiment,
        "Epochs": params["epochs"],
        "Batch Size": params["batch"],
        "Optimizer": optimizer,
        "Image Size": params["imgsz"],
        "Learning Rate": lr0,
        "Training Time (s)": round(train_time, 2),
        "Precision": results.results_dict["metrics/precision(B)"],
        "Recall": results.results_dict["metrics/recall(B)"],
        "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
        "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        "Parameters": json.dumps(params)
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# =============================
# Rerun only RMSProp with lower LR
# =============================

print("\n🚀 Re-running Case 6: RMSProp with lr0=0.001...")
model = YOLO("yolov8s.pt")

start_time = time.time()
results = model.train(
    data=data_yaml,
    epochs=base_params["epochs"],
    batch=base_params["batch"],
    lr0=0.001,                   # 🔹 lower learning rate for RMSProp
    optimizer="RMSProp",
    imgsz=base_params["imgsz"],
    weight_decay=base_params["weight_decay"],
    dropout=base_params["dropout"],
    freeze=base_params["freeze"]
)
end_time = time.time()

log_result("case6_opt_RMSProp_fixed", "RMSProp", 0.001, results, end_time - start_time)

print("\n✅ RMSProp re-run completed and logged with lr0=0.001")



🚀 Re-running Case 6: RMSProp with lr0=0.001...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms=False, opset=None, optimize=False, op

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 796.13it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 43.1±21.6 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 498.72it/s]

...

      32/32      7.89G      1.247     0.6023      1.397        142        640: 100%|██████████| 29/29 [00:15<00:00,  1.89it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.69it/s]

                   all        230       1440      0.947      0.957      0.978      0.607



32 epochs completed in 0.165 hours.
Optimizer stripped from runs/detect/train6/weights/last.pt, 22.5MB
Optimizer stripped from runs/detect/train6/weights/best.pt, 22.5MB

Validating runs/detect/train6/weights/best.pt...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:03<00:00,  1.32it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.958      0.956      0.979      0.653
     DC Voltage Source        103        144      0.939      0.972       0.98      0.737
              Resistor        183        605      0.976      0.988      0.986      0.616
        Current Source        113        150      0.903      0.993       0.96      0.697
              Inductor        117        177      0.956      0.938      0.986      0.606
             Capacitor        115        184      0.982      0.878      0.975      0.557
     AC Voltage Source         65        180      0.989      0.964      0.989      0.701
Speed: 0.1ms preprocess, 3.6ms inference, 0.0ms loss, 6.1ms postprocess per image
Results saved to runs/detect/train6

✅ RMSProp re-run completed and logged with lr0=0.001


In [2]:
from ultralytics import YOLO
import time, pandas as pd, os, json
from tabulate import tabulate

# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# baseline params (fixed from previous bests)
base_params = {
    "batch": 32,
    "epochs": 32,
    "optimizer": "AdamW",   # chosen from Case 6
    "weight_decay": 0.001,
    "dropout": 0.0,
    "freeze": 0,
    "activation": "SiLU",
    "imgsz": 640
}

# results log file (big one file for all cases)
results_file = "case1_results.csv"

# ensure CSV exists
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", 
        "Epochs", 
        "Batch Size",
        "Optimizer",
        "Image Size",
        "Learning Rate",
        "Training Time (s)", 
        "Precision", 
        "Recall", 
        "mAP@0.5", 
        "mAP@0.5:0.95", 
        "Parameters"
    ]).to_csv(results_file, index=False)

def log_result(experiment, lr0, results, train_time):
    """Append YOLO training results to CSV"""
    df = pd.read_csv(results_file)

    params = base_params.copy()
    params["lr0"] = lr0

    new_row = {
        "Experiment": experiment,
        "Epochs": params["epochs"],
        "Batch Size": params["batch"],
        "Optimizer": params["optimizer"],
        "Image Size": params["imgsz"],
        "Learning Rate": lr0,
        "Training Time (s)": round(train_time, 2),
        "Precision": results.results_dict["metrics/precision(B)"],
        "Recall": results.results_dict["metrics/recall(B)"],
        "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
        "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        "Parameters": json.dumps(params)
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# =============================
# Case 7: Learning Rate variations
# =============================

lr_variations = [0.001, 0.005, 0.01, 0.05, 0.1]
all_results = []

for lr in lr_variations:
    print(f"\n🚀 Case 7: Training YOLOv8s with optimizer={base_params['optimizer']}, lr0={lr}, epochs={base_params['epochs']}, batch={base_params['batch']}, imgsz={base_params['imgsz']}...")
    model = YOLO("yolov8s.pt")

    start_time = time.time()
    results = model.train(
        data=data_yaml,
        epochs=base_params["epochs"],
        batch=base_params["batch"],
        lr0=lr,
        optimizer=base_params["optimizer"],
        imgsz=base_params["imgsz"],
        weight_decay=base_params["weight_decay"],
        dropout=base_params["dropout"],
        freeze=base_params["freeze"]
    )
    end_time = time.time()

    log_result(f"case7_lr_{lr}", lr, results, end_time - start_time)

    all_results.append([
        f"case7_lr_{lr}",
        base_params["epochs"],
        base_params["batch"],
        base_params["optimizer"],
        base_params["imgsz"],
        lr,
        round(end_time - start_time, 2),
        results.results_dict["metrics/precision(B)"],
        results.results_dict["metrics/recall(B)"],
        results.results_dict["metrics/mAP50(B)"],
        results.results_dict["metrics/mAP50-95(B)"],
    ])

# =============================
# Print pretty results table
# =============================
headers = ["Experiment", "Epochs", "Batch Size", "Optimizer", "Image Size", "Learning Rate", "Train Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"]
print("\n📊 Case 7 Results Table:\n")
print(tabulate(all_results, headers=headers, tablefmt="grid"))

print(f"\n✅ Case 7 completed. Results appended to {results_file}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🚀 Case 7: Training YOLOv8s with optimizer=AdamW, lr0=0.001, epochs=32, batch=32, imgsz=640...


Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, p

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7.9±5.8 MB/s, size: 51.9 KB)


train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:04<00:00, 229.66it/s]


...

      32/32      7.91G      1.256     0.6273      1.284        142        640: 100%|██████████| 29/29 [00:15<00:00,  1.85it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.50it/s]

                   all        230       1440      0.952      0.956      0.979      0.631



32 epochs completed in 0.161 hours.
Optimizer stripped from runs/detect/train5/weights/last.pt, 22.5MB
Optimizer stripped from runs/detect/train5/weights/best.pt, 22.5MB

Validating runs/detect/train5/weights/best.pt...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:03<00:00,  1.16it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.965      0.941      0.978      0.634
     DC Voltage Source        103        144      0.945      0.951      0.972      0.702
              Resistor        183        605      0.982       0.98      0.984      0.603
        Current Source        113        150      0.949        0.9      0.975      0.683
              Inductor        117        177      0.982      0.925      0.987      0.567
             Capacitor        115        184      0.971      0.916      0.975      0.568
     AC Voltage Source         65        180       0.96      0.972      0.977      0.679
Speed: 0.2ms preprocess, 3.7ms inference, 0.0ms loss, 3.8ms postprocess per image
Results saved to runs/detect/train5

📊 Case 7 Results Table:

+----------------+----------+--------------+-------------+--------------+-----------------+------------------+-------------+----------+-----------+----------------+
| Experiment     |   Epochs |   Batch Size | Optimizer   |   Image S

In [3]:
from ultralytics import YOLO
import time, pandas as pd, os, json
from tabulate import tabulate

# dataset path
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# baseline params (fixed from previous bests)
base_params = {
    "batch": 32,
    "epochs": 32,
    "optimizer": "AdamW",
    "lr0": 0.001,          # chosen from Case 7
    "weight_decay": 0.001,
    "dropout": 0.0,
    "activation": "SiLU",
    "imgsz": 640
}

# results log file (master file for all cases)
results_file = "case1_results.csv"

# ensure CSV exists
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", 
        "Epochs", 
        "Batch Size",
        "Optimizer",
        "Image Size",
        "Learning Rate",
        "Freeze Layers",
        "Training Time (s)", 
        "Precision", 
        "Recall", 
        "mAP@0.5", 
        "mAP@0.5:0.95", 
        "Parameters"
    ]).to_csv(results_file, index=False)

def log_result(experiment, freeze_layers, results, train_time):
    """Append YOLO training results to CSV"""
    df = pd.read_csv(results_file)

    params = base_params.copy()
    params["freeze"] = freeze_layers

    new_row = {
        "Experiment": experiment,
        "Epochs": params["epochs"],
        "Batch Size": params["batch"],
        "Optimizer": params["optimizer"],
        "Image Size": params["imgsz"],
        "Learning Rate": params["lr0"],
        "Freeze Layers": freeze_layers,
        "Training Time (s)": round(train_time, 2),
        "Precision": results.results_dict["metrics/precision(B)"],
        "Recall": results.results_dict["metrics/recall(B)"],
        "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
        "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        "Parameters": json.dumps(params)
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# =============================
# Case 8: Freeze Layer variations
# =============================

freeze_variations = [0, 10, 15, 22]
all_results = []

for frz in freeze_variations:
    print(f"\n🚀 Case 8: Training YOLOv8s with freeze={frz}, optimizer={base_params['optimizer']}, lr0={base_params['lr0']}, epochs={base_params['epochs']}...")
    model = YOLO("yolov8s.pt")

    start_time = time.time()
    results = model.train(
        data=data_yaml,
        epochs=base_params["epochs"],
        batch=base_params["batch"],
        lr0=base_params["lr0"],
        optimizer=base_params["optimizer"],
        imgsz=base_params["imgsz"],
        weight_decay=base_params["weight_decay"],
        dropout=base_params["dropout"],
        freeze=frz   # 🔹 freezing layers here
    )
    end_time = time.time()

    log_result(f"case8_freeze_{frz}", frz, results, end_time - start_time)

    all_results.append([
        f"case8_freeze_{frz}",
        base_params["epochs"],
        base_params["batch"],
        base_params["optimizer"],
        base_params["imgsz"],
        base_params["lr0"],
        frz,
        round(end_time - start_time, 2),
        results.results_dict["metrics/precision(B)"],
        results.results_dict["metrics/recall(B)"],
        results.results_dict["metrics/mAP50(B)"],
        results.results_dict["metrics/mAP50-95(B)"],
    ])

# =============================
# Print pretty results table
# =============================
headers = ["Experiment", "Epochs", "Batch Size", "Optimizer", "Image Size", "Learning Rate", "Freeze Layers", "Train Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"]
print("\n📊 Case 8 Results Table:\n")
print(tabulate(all_results, headers=headers, tablefmt="grid"))

print(f"\n✅ Case 8 completed. Results appended to {results_file}")



🚀 Case 8: Training YOLOv8s with freeze=0, optimizer=AdamW, lr0=0.001, epochs=32...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms=F

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 675.43it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 40.5±22.1 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 413.13it/s]

...

      32/32      4.68G      1.242     0.6318      1.376        142        640: 100%|██████████| 29/29 [00:07<00:00,  3.90it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:02<00:00,  1.91it/s]

                   all        230       1440      0.969       0.97      0.984      0.636



32 epochs completed in 0.093 hours.
Optimizer stripped from runs/detect/train9/weights/last.pt, 22.5MB
Optimizer stripped from runs/detect/train9/weights/best.pt, 22.5MB

Validating runs/detect/train9/weights/best.pt...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:03<00:00,  1.17it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.971      0.975      0.985      0.637
     DC Voltage Source        103        144      0.974      0.958      0.978      0.716
              Resistor        183        605      0.971      0.988      0.987      0.596
        Current Source        113        150       0.97      0.993      0.989      0.695
              Inductor        117        177      0.949      0.949      0.979      0.577
             Capacitor        115        184      0.968      0.978      0.983      0.549
     AC Voltage Source         65        180      0.994      0.984      0.994      0.689
Speed: 0.1ms preprocess, 3.7ms inference, 0.0ms loss, 4.5ms postprocess per image
Results saved to runs/detect/train9

📊 Case 8 Results Table:

+-----------------+----------+--------------+-------------+--------------+-----------------+-----------------+------------------+-------------+----------+-----------+----------------+
| Experiment      |   Epochs |   Batch Size | Opt

In [5]:
import yaml
from ultralytics.nn.tasks import yaml_model_load
import time, pandas as pd
from ultralytics import YOLO
from tabulate import tabulate

# master results file
results_file = "case1_results.csv"

# dataset
data_yaml = "/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml"

# baseline params (fixed)
base_params = {
    "epochs": 32,
    "batch": 32,
    "optimizer": "AdamW",      # use AdamW from Case 6
    "lr0": 0.001,              # best from Case 7
    "weight_decay": 0.001,     # best from Case 2
    "dropout": 0.0,
    "imgsz": 640,
    "freeze": 0                # train all layers
}

# Load base YOLOv8n config (you can switch to yolov8s.yaml if needed)
cfg = yaml_model_load("yolov8s.yaml")

def create_yaml_with_activation(act_expr):
    cfg_mod = cfg.copy()
    cfg_mod['activation'] = act_expr
    out_path = f"/kaggle/working/yolov8s_{act_expr.replace('nn.','').replace('()','').replace('0.1','01').lower()}.yaml"
    with open(out_path, "w") as f:
        yaml.dump(cfg_mod, f)
    print(f"✅ Created {out_path}")
    return out_path

# Create activation-specific YAMLs
relu_yaml = create_yaml_with_activation("nn.ReLU()")
leaky_yaml = create_yaml_with_activation("nn.LeakyReLU(0.1)")
mish_yaml = create_yaml_with_activation("nn.Mish()")
silu_yaml = create_yaml_with_activation("nn.SiLU()")  # baseline

activations = {
    "ReLU": relu_yaml,
    "LeakyReLU": leaky_yaml,
    "Mish": mish_yaml,
    "SiLU": silu_yaml
}

# =============================
# Case 9: Activation Functions
# =============================
all_results = []

for act, yaml_path in activations.items():
    print(f"\n🚀 Training with {act} activation...")

    model = YOLO(yaml_path)

    start = time.time()
    results = model.train(
        data=data_yaml,
        epochs=base_params["epochs"],
        batch=base_params["batch"],
        lr0=base_params["lr0"],
        optimizer=base_params["optimizer"],
        imgsz=base_params["imgsz"],
        weight_decay=base_params["weight_decay"],
        dropout=base_params["dropout"],
        freeze=base_params["freeze"]
    )
    end = time.time()

    # validate
    val = model.val()

    result = {
        "Experiment": f"case9_act_{act}",
        "Epochs": base_params["epochs"],
        "Batch Size": base_params["batch"],
        "Optimizer": base_params["optimizer"],
        "Image Size": base_params["imgsz"],
        "Learning Rate": base_params["lr0"],
        "Activation": act,
        "Freeze Layers": base_params["freeze"],
        "Training Time (s)": round(end - start, 2),
        "Precision": val.results_dict.get("metrics/precision(B)", val.results_dict.get("metrics/precision", None)),
        "Recall": val.results_dict.get("metrics/recall(B)", val.results_dict.get("metrics/recall", None)),
        "mAP@0.5": val.results_dict.get("metrics/mAP50(B)", val.results_dict.get("metrics/mAP_0.5", None)),
        "mAP@0.5:0.95": val.results_dict.get("metrics/mAP50-95(B)", val.results_dict.get("metrics/mAP_0.5:0.95", None))
    }

    # append to CSV
    df = pd.read_csv(results_file)
    df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
    df.to_csv(results_file, index=False)

    all_results.append(result)

# Show results
print("\n📊 Case 9 Results Table:\n")
print(tabulate(pd.DataFrame(all_results), headers="keys", tablefmt="grid"))


✅ Created /kaggle/working/yolov8s_relu.yaml
✅ Created /kaggle/working/yolov8s_leakyrelu(01).yaml
✅ Created /kaggle/working/yolov8s_mish.yaml
✅ Created /kaggle/working/yolov8s_silu.yaml

🚀 Training with ReLU activation...
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 861.11it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 52.3±25.1 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 456.40it/s]


...

                   all        230       1440      0.986      0.984      0.987      0.679
     DC Voltage Source        103        144      0.993      0.967       0.98      0.752
              Resistor        183        605      0.984      0.985      0.989      0.644
        Current Source        113        150      0.977          1      0.986      0.737
              Inductor        117        177      0.978      0.988      0.989      0.615
             Capacitor        115        184      0.991      0.984      0.986      0.604
     AC Voltage Source         65        180      0.994      0.981      0.992      0.721
Speed: 0.1ms preprocess, 3.8ms inference, 0.0ms loss, 6.1ms postprocess per image
Results saved to runs/detect/train13
Ultralytics 8.3.184 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLOv8s_silu summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 84.6±64.8 MB/s, size: 58

val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 826.18it/s]

WARNING ⚠️ val: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/val is not writeable, cache not saved.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:04<00:00,  1.87it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.986      0.984      0.987       0.68
     DC Voltage Source        103        144      0.993      0.967       0.98      0.756
              Resistor        183        605      0.984      0.987      0.989      0.645
        Current Source        113        150      0.977          1      0.986      0.739
              Inductor        117        177      0.978      0.988       0.99      0.615
             Capacitor        115        184      0.991      0.984      0.987      0.606
     AC Voltage Source         65        180      0.994      0.981      0.992      0.721
Speed: 2.2ms preprocess, 5.4ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to runs/detect/train132

📊 Case 9 Results Table:

+----+---------------------+----------+--------------+-------------+--------------+-----------------+--------------+-----------------+---------------------+-------------+----------+-----------+----------------+
|    | Experiment   

In [6]:
import pandas as pd

# Read your full results file
df = pd.read_csv("case1_results.csv")

# Ensure all important columns exist (fill with defaults if missing)
for col in ["Case", "Index", "Dropout", "Activation", "Observation"]:
    if col not in df.columns:
        df[col] = None

# Assign case numbers based on Experiment naming
def assign_case(exp):
    if "case1_" in exp: return "Case 1: Epoch Variations"
    if "case2_" in exp: return "Case 2: Weight Decay"
    if "case3_" in exp: return "Case 3: Batch Size"
    if "case4_" in exp: return "Case 4: Dropout"
    if "case5_" in exp: return "Case 5: Image Size"
    if "case6_" in exp: return "Case 6: Optimizers"
    if "case7_" in exp: return "Case 7: Learning Rates"
    if "case8_" in exp: return "Case 8: Freeze Layers"
    if "case9_" in exp: return "Case 9: Activation Functions"
    return "Unknown"

df["Case"] = df["Experiment"].apply(assign_case)

# Add sequential index inside each case
df["Index"] = df.groupby("Case").cumcount() + 1

# Fill observations manually (example rules, you can refine later)
def add_observation(row):
    if row["Case"] == "Case 7: Learning Rates":
        if row["Learning Rate"] == 0.001: return "Best Generalization, Balanced"
        if row["Learning Rate"] == 0.005: return "Best Recall & mAP@0.5"
        if row["Learning Rate"] == 0.01: return "Best Precision"
        if row["Learning Rate"] >= 0.05: return "Performance Dropped"
    if row["Case"] == "Case 8: Freeze Layers":
        if row["Freeze Layers"] == 0: return "Best Generalization, Training Slow"
        if row["Freeze Layers"] == 10: return "Best mAP@0.5, Training Faster"
        if row["Freeze Layers"] == 15: return "Good Precision, Training Faster"
        if row["Freeze Layers"] == 22: return "Performance Dropped, Training Fastest"
    if row["Case"] == "Case 9: Activation Functions":
        if row["Activation"] == "SiLU": return "Best Overall, Default Choice"
        else: return "Performance Lower than SiLU"
    return "Balanced"

df["Observation"] = df.apply(add_observation, axis=1)

# Reorder columns for neatness
cols = ["Case","Index","Experiment","Epochs","Batch Size","Learning Rate","weight_decay",
        "Optimizer","Image Size","Dropout","Freeze Layers","Activation",
        "Training Time (s)","Precision","Recall","mAP@0.5","mAP@0.5:0.95","Observation"]

final_df = df[[c for c in cols if c in df.columns]]

# Save consolidated CSV
final_df.to_csv("all_cases_results.csv", index=False)

print("✅ Consolidated CSV saved as all_cases_results.csv")


✅ Consolidated CSV saved as all_cases_results.csv
